In [2]:
import pandas as pd 
import numpy as np 
from sklearn.model_selection import train_test_split 
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix ,recall_score , precision_score , f1_score , accuracy_score, classification_report

data  = pd.read_csv("../data/Telco-Customer-Churn.csv")

customer_ids = data["customerID"]

data = data.drop("customerID", axis=1)

data = pd.get_dummies(data, drop_first=True)


X = data.drop("Churn_Yes", axis=1)
y = data["Churn_Yes"]

X_train,X_test,y_train,y_test = train_test_split(X , y , test_size = 0.2 , random_state = 1 , stratify=y)

model = XGBClassifier(n_estimators = 100 , random_state = 1)
model.fit(X_train , y_train)

y_pred_xgb = model.predict(X_test)
print(y_pred_xgb[:10])

print("accuracy :" , accuracy_score(y_test , y_pred_xgb))
print("precision :" ,precision_score(y_test , y_pred_xgb))
print("recall :" , recall_score(y_test , y_pred_xgb))
print("f1 score :" , f1_score(y_test , y_pred_xgb))
print("\n classification report :")
print(classification_report(y_test , y_pred_xgb))
print("\n confusion matrix :")
print(confusion_matrix(y_test , y_pred_xgb))


[0 1 0 0 0 1 0 0 1 0]
accuracy : 0.7927608232789212
precision : 0.6331168831168831
recall : 0.5213903743315508
f1 score : 0.5718475073313783

 classification report :
              precision    recall  f1-score   support

       False       0.84      0.89      0.86      1035
        True       0.63      0.52      0.57       374

    accuracy                           0.79      1409
   macro avg       0.74      0.71      0.72      1409
weighted avg       0.78      0.79      0.79      1409


 confusion matrix :
[[922 113]
 [179 195]]


In [3]:
from sklearn.model_selection import RandomizedSearchCV

params = {
   "n_estimators": [50, 100, 200],     
   "max_depth": [2, 5, 10],     
   "min_child_weight": [1, 3, 5],     
   "learning_rate": [0.01, 0.1, 0.2],     
   "subsample": [0.8, 1.0] 
}

random_search = RandomizedSearchCV(XGBClassifier(n_estimators = 100 ,random_state = 0),param_distributions=params,n_iter=20,cv=5,scoring="f1",random_state=1,n_jobs=-1)
random_search.fit(X_train , y_train)
print(random_search.best_estimator_)
print(random_search.best_score_)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.2, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=2,
              max_leaves=None, min_child_weight=3, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)
0.5901644478683948


In [4]:
xgb_model = random_search.best_estimator_
y_pred_xgb = xgb_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb))
print("Recall:", recall_score(y_test, y_pred_xgb))
print("F1 Score:", f1_score(y_test, y_pred_xgb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

Accuracy: 0.7998580553584103
Precision: 0.6619718309859155
Recall: 0.5026737967914439
F1 Score: 0.5714285714285714

Classification Report:
              precision    recall  f1-score   support

       False       0.83      0.91      0.87      1035
        True       0.66      0.50      0.57       374

    accuracy                           0.80      1409
   macro avg       0.75      0.70      0.72      1409
weighted avg       0.79      0.80      0.79      1409


Confusion Matrix:
[[939  96]
 [186 188]]


In [5]:
import joblib
joblib.dump(xgb_model ,"../models/xgboost.pkl")

['../models/xgboost.pkl']